In [ ]:
# Run once if a library is missing:
# !pip install rasterio geopandas matplotlib pandas numpy

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.plot import plotting_extent
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path("Data")          # folder holding the 5 exported files
plt.rcParams["figure.dpi"] = 110

for f in sorted(DATA.glob("*")):
    print(f.name)

In [ ]:
from pathlib import Path

DATA = Path(".")

files = (
    list(DATA.glob("*.tif")) +
    list(DATA.glob("*.csv")) +
    list(DATA.glob("*.geojson"))
)

for f in sorted(files):
    print(f.name)

In [ ]:
jul_path = DATA / "SriLanka_NO2_Jul2025.tif"
nov_path = DATA / "SriLanka_NO2_Nov2025.tif"

with rasterio.open(jul_path) as src:
    print("CRS        :", src.crs)
    print("Size       :", src.width, "x", src.height, "pixels")
    print("Bands      :", src.count)
    print("Pixel size :", src.res, "(degrees)")
    print("Bounds     :", src.bounds)
    print("No-data    :", src.nodata)
    print("Data type  :", src.dtypes[0])

In [ ]:
jul_path = DATA / "Eastern_NO2_Jul2025.tif"
nov_path = DATA / "Eastern_NO2_Nov2025.tif"

with rasterio.open(jul_path) as src:
    print("CRS        :", src.crs)
    print("Size       :", src.width, "x", src.height, "pixels")
    print("Bands      :", src.count)
    print("Pixel size :", src.res, "(degrees)")
    print("Bounds     :", src.bounds)
    print("No-data    :", src.nodata)
    print("Data type  :", src.dtypes[0])

In [ ]:
jul_path = DATA / "Eastern_NO2_Jul2025.tif"
nov_path = DATA / "Eastern_NO2_Nov2025.tif"

with rasterio.open(jul_path) as src:
    print("CRS        :", src.crs)
    print("Size       :", src.width, "x", src.height, "pixels")
    print("Bands      :", src.count)
    print("Pixel size :", src.res, "(degrees)")
    print("Bounds     :", src.bounds)
    print("No-data    :", src.nodata)
    print("Data type  :", src.dtypes[0])

In [ ]:
NODATA = -9999
TO_UMOL = 1e6   # mol/m² -> µmol/m²

def read_no2(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float64")
        extent = plotting_extent(src)

    # Handle NoData values
    arr[arr == NODATA] = np.nan

    # Convert from mol/m² to µmol/m²
    return arr * TO_UMOL, extent

# Read July and November rasters
jul, extent = read_no2(DATA / "SriLanka_NO2_Jul2025.tif")
nov, _ = read_no2(DATA / "SriLanka_NO2_Nov2025.tif")

# Summary statistics
print("July  : min %.1f  max %.1f µmol/m²" % (np.nanmin(jul), np.nanmax(jul)))
print("Nov   : min %.1f  max %.1f µmol/m²" % (np.nanmin(nov), np.nanmax(nov)))
print("No-data pixels — July:", np.isnan(jul).sum(),
      " Nov:", np.isnan(nov).sum())

In [ ]:
# Load Eastern Province boundary
eastern = gpd.read_file(DATA / "Eastern_boundary.geojson").to_crs("EPSG:4326")

# Same colour scale for both months
vmin = 0
vmax = np.nanpercentile(
    np.concatenate([jul.ravel(), nov.ravel()]), 98
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

for ax, arr, title in zip(
    axes,
    [jul, nov],
    ["July 2025", "November 2025"]
):
    im = ax.imshow(
        arr,
        extent=extent,
        cmap="YlOrRd",
        vmin=vmin,
        vmax=vmax
    )

    eastern.boundary.plot(ax=ax, color="black", linewidth=1.2)

    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Longitude (°E)")
    ax.set_ylabel("Latitude (°N)")

fig.colorbar(
    im,
    ax=axes,
    shrink=0.85,
    label="Tropospheric NO₂ column (µmol/m²)"
)

fig.suptitle(
    "Sentinel-5P TROPOMI Monthly Mean NO₂ - Eastern Province, Sri Lanka",
    fontsize=13
)

plt.show()

In [ ]:
srilanka = gpd.read_file(DATA / "SriLanka_provinces.geojson").to_crs("EPSG:4326")

In [ ]:
eastern.boundary.plot(ax=ax, color="black", linewidth=1.2)

In [ ]:
srilanka.boundary.plot(ax=ax, color="black", linewidth=0.8)

In [ ]:
def clip_to_eastern(path):
    with rasterio.open(path) as src:
        out, _ = mask(
            src,
            eastern.geometry,
            crop=True,
            nodata=NODATA,
            filled=True
        )

    arr = out[0].astype("float64")
    arr[arr == NODATA] = np.nan

    return arr * TO_UMOL

# Clip July and November rasters
jul_d = clip_to_eastern(DATA / "Eastern_NO2_Jul2025.tif")
nov_d = clip_to_eastern(DATA / "Eastern_NO2_Nov2025.tif")

# Summary statistics
def summary(arr):
    v = arr[~np.isnan(arr)]
    return {
        "pixels": v.size,
        "mean": v.mean(),
        "median": np.median(v),
        "min": v.min(),
        "max": v.max(),
        "std": v.std()
    }

stats = pd.DataFrame({
    "July 2025": summary(jul_d),
    "Nov 2025": summary(nov_d)
}).T.round(1)

stats

In [ ]:
pct = (
    (stats.loc["Nov 2025", "mean"] - stats.loc["July 2025", "mean"])
    / stats.loc["July 2025", "mean"]
    * 100
)

ratio = (
    stats.loc["Nov 2025", "mean"]
    / stats.loc["July 2025", "mean"]
)

print(f"Mean NO₂ over Eastern Province changed by {pct:+.1f}% from July to November")
print(f"November is {ratio:.1f}× the July level")

In [ ]:
def summary(arr):
    v = arr[~np.isnan(arr)]
    return {
        "pixels": v.size,
        "mean": v.mean(),
        "median": np.median(v),
        "min": v.min(),
        "max": v.max(),
        "std": v.std()
    }

stats = pd.DataFrame({
    "July 2025": summary(jul_d),
    "Nov 2025": summary(nov_d)
}).T.round(1)

print(stats)


In [ ]:
pct = (
    (stats.loc["Nov 2025", "mean"] - stats.loc["July 2025", "mean"])
    / stats.loc["July 2025", "mean"]
    * 100
)

ratio = (
    stats.loc["Nov 2025", "mean"]
    / stats.loc["July 2025", "mean"]
)

print(f"Mean NO₂ over Eastern Province changed by {pct:+.1f}% from July to November")
print(f"November is {ratio:.1f}× the July level")

In [ ]:
eastern = gpd.read_file(DATA / "Eastern_boundary.geojson").to_crs("EPSG:4326")

In [ ]:
# Difference map (November - July)
diff = nov - jul

# Symmetric colour limits
lim = np.nanpercentile(np.abs(diff), 98)

fig, ax = plt.subplots(figsize=(6.5, 5.5))

im = ax.imshow(
    diff,
    extent=extent,
    cmap="RdBu_r",
    vmin=-lim,
    vmax=lim
)

# Plot Eastern Province boundary
eastern.boundary.plot(
    ax=ax,
    color="black",
    linewidth=1.2
)

ax.set_title("Change in NO₂: November − July 2025")
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")

fig.colorbar(
    im,
    ax=ax,
    label="Δ NO₂ (µmol/m²)"
)

plt.show()

In [ ]:
srilanka = gpd.read_file("SriLanka_provinces.geojson").to_crs("EPSG:4326")